# LangChain: The Orchestration Engine Powering Modern LLM Applications
---

## Objective
This notebook demonstrates the core components of LangChain with working Python code examples.
It covers: LLMs, PromptTemplates, Chains, Memory, Agents, Tools, Document Loaders, Vector Stores, and Real-World Use Cases.

---

## Step 0 — Install Required Libraries
Run this cell once to install all dependencies needed for this notebook.

In [1]:
# Install all required packages
!pip install langchain langchain-openai langchain-community langchain-core
!pip install openai faiss-cpu tiktoken
!pip install pypdf duckduckgo-search wikipedia
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.0 which is incompatible.



  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 1.2 MB/s eta 0:00:02
   ---------------- ----------------------- 1.0/2.5 MB 1.7 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.5 MB 2.0 MB/s eta 0:00:01
   --------------------------------- ------ 2.1/2.5 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 2.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.2 MB 2.8 MB/s eta 0:00:01
   ------------------------------------ -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.0 which is incompatible.


Defaulting to user installation because normal site-packages is not writeable


---
## Step 1 — Setup: API Key Configuration

Set your OpenAI API key here. All code blocks below will use this key.
You can get a free/paid key from: https://platform.openai.com/api-keys

In [2]:
import os

# ------------------------------------------------------------
# SET YOUR OPENAI API KEY HERE
# ------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
print("API Key configured successfully!")

API Key configured successfully!


---
## Component 1 — LLMs and Chat Models

**What it is:** LangChain wraps any LLM (OpenAI, HuggingFace, Anthropic) behind a unified interface.  
**Why it exists:** So you can swap models without rewriting all your API calls.

### Architecture Flow:
```
User Message → ChatOpenAI (LLM Wrapper) → AI Response
```

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ── Initialize the LLM wrapper ──────────────────────────────────
llm = ChatOpenAI(
    model="gpt-3.5-turbo",   # model name
    temperature=0.7,          # creativity: 0 = deterministic, 1 = creative
    api_key=OPENAI_API_KEY
)

# ── Build a message list (System + Human) ───────────────────────
messages = [
    SystemMessage(content="You are a helpful data science tutor. Be concise."),
    HumanMessage(content="Explain what a vector embedding is in 2 sentences.")
]

# ── Invoke the LLM ──────────────────────────────────────────────
response = llm.invoke(messages)

print("=== LLM Response ===")
print(response.content)
print(f"\nModel used: {response.response_metadata.get('model_name', 'gpt-3.5-turbo')}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

---
## Component 2 — Prompts and Prompt Templates

**What it is:** Reusable, parameterized templates for constructing prompts dynamically.  
**Why it exists:** In production, prompts are never static — you need to inject variables, context, and user input cleanly.

### Architecture Flow:
```
Variables → PromptTemplate.format() → Filled Prompt String → LLM
```

In [5]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# ── Example 1: Simple PromptTemplate ────────────────────────────
simple_template = PromptTemplate(
    input_variables=["topic", "level"],
    template="Explain {topic} to a {level} level student in 3 bullet points."
)

# Fill in the variables
filled = simple_template.format(topic="neural networks", level="beginner")
print("=== Filled Simple Template ===")
print(filled)

# ── Example 2: ChatPromptTemplate (for multi-turn chat) ──────────
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise and precise."),
    ("human", "What is {concept}?")
])

# Format into LangChain message objects
messages = chat_template.format_messages(
    domain="machine learning",
    concept="gradient descent"
)

print("\n=== Chat Prompt Messages ===")
for msg in messages:
    print(f"[{msg.__class__.__name__}]: {msg.content}")

# ── Send the chat prompt to the LLM ─────────────────────────────
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)
response = llm.invoke(messages)
print("\n=== LLM Response ===")
print(response.content)

=== Filled Simple Template ===
Explain neural networks to a beginner level student in 3 bullet points.

=== Chat Prompt Messages ===
[SystemMessage]: You are an expert in machine learning. Be concise and precise.
[HumanMessage]: What is gradient descent?


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

---
## Component 3 — Chains (LCEL — LangChain Expression Language)

**What it is:** Chains connect components using the `|` pipe operator. Output of one flows as input to the next.  
**Why it exists:** Real applications need multi-step pipelines — format prompt → call LLM → parse output.

### Architecture Flow:
```
Input → PromptTemplate | LLM | OutputParser → Final String
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Define individual components ─────────────────────────────────
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3, api_key=OPENAI_API_KEY)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python coding assistant."),
    ("human", "Write a Python function to {task}. Keep it short and well-commented.")
])

# StrOutputParser converts AIMessage object → plain string
parser = StrOutputParser()

# ── Build the chain using LCEL pipe operator ─────────────────────
# Chain: prompt → llm → parser
chain = prompt | llm | parser

# ── Invoke the chain ─────────────────────────────────────────────
result = chain.invoke({"task": "calculate the factorial of a number recursively"})

print("=== Chain Output ===")
print(result)

In [ ]:
# ── Multi-step Chain: Summarize → Translate ───────────────────────
# Step 1: Summarize the input text
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this in one sentence: {text}"
)

# Step 2: Translate the summary to French
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this English text to French: {text}"
)

summarize_chain = summarize_prompt | llm | parser
translate_chain  = translate_prompt | llm | parser

# Combine: output of summarize becomes input of translate
combined_chain = (
    summarize_chain
    | (lambda text: {"text": text})   # convert string → dict for next prompt
    | translate_chain
)

input_text = "LangChain is a powerful framework for building LLM-powered applications using modular components."
result = combined_chain.invoke({"text": input_text})

print("=== Original Text ===")
print(input_text)
print("\n=== Summarized + Translated to French ===")
print(result)

---
## Component 4 — Memory

**What it is:** Memory modules preserve conversation history across turns so the LLM can reference prior messages.  
**Why it exists:** Without memory, every LLM call is stateless — the bot forgets everything after each response.

### Architecture Flow:
```
User Message → [Memory appends history] → Prompt (with history) → LLM → Response → [Saved to Memory]
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)

# ── ConversationBufferMemory stores ALL past messages ────────────
memory = ConversationBufferMemory()

# ── ConversationChain wraps LLM + Memory together ────────────────
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False   # set True to see the full prompt with history printed
)

# ── Simulate a multi-turn conversation ───────────────────────────
print("=== Turn 1 ===")
r1 = conversation.predict(input="Hi! My name is Pranav and I am learning LangChain.")
print(f"AI: {r1}")

print("\n=== Turn 2 ===")
r2 = conversation.predict(input="What is my name and what am I learning?")
print(f"AI: {r2}")
# The LLM correctly remembers 'Pranav' and 'LangChain' from Turn 1

print("\n=== Turn 3 ===")
r3 = conversation.predict(input="Can you suggest me what to learn next after LangChain?")
print(f"AI: {r3}")

# ── View stored memory ───────────────────────────────────────────
print("\n=== Stored Memory (all messages) ===")
for msg in memory.chat_memory.messages:
    print(f"[{msg.__class__.__name__}]: {msg.content[:80]}...")

In [ ]:
# ── Window Memory: Keeps only last k=2 exchanges ─────────────────
# Useful for long conversations to avoid exceeding token limits
from langchain.memory import ConversationBufferWindowMemory

window_memory = ConversationBufferWindowMemory(k=2)  # only last 2 exchanges kept

window_conv = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

window_conv.predict(input="My favorite color is blue.")
window_conv.predict(input="I love playing chess.")
window_conv.predict(input="I am from Hyderabad.")

# After 3 turns with k=2, turn 1 ('blue') is dropped from memory
r = window_conv.predict(input="What is my favorite color? And where am I from?")
print("=== Window Memory Response ===")
print(r)
print("(Note: favorite color may be forgotten since k=2 keeps only last 2 turns)")

---
## Component 5 — Agents (ReAct Framework)

**What it is:** An Agent lets the LLM autonomously decide which tools to call and in what order using the ReAct (Reason + Act) loop.  
**Why it exists:** Some tasks require dynamic, multi-step tool usage that cannot be hardcoded into a fixed chain.

### Architecture Flow:
```
User Query → Agent (LLM) → Thought → Action (Tool Call) → Observation → Thought → ... → Final Answer
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import tool
from langchain import hub
import math

# ── Define Custom Tools ──────────────────────────────────────────
@tool
def scientific_calculator(expression: str) -> str:
    """
    Evaluates mathematical expressions safely.
    Supports: +, -, *, /, **, sqrt, sin, cos, log, pi, e
    Example input: 'sqrt(144) + log(100)'
    """
    try:
        # Only allow math module functions — no dangerous builtins
        safe_dict = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"


@tool
def unit_converter(query: str) -> str:
    """
    Converts common units. Format: 'value from_unit to to_unit'
    Supports: km/miles, kg/lbs, celsius/fahrenheit
    Example: '100 celsius to fahrenheit'
    """
    parts = query.lower().split()
    try:
        value = float(parts[0])
        from_unit, to_unit = parts[1], parts[3]

        conversions = {
            ("km", "miles"):          lambda x: x * 0.621371,
            ("miles", "km"):          lambda x: x * 1.60934,
            ("kg", "lbs"):            lambda x: x * 2.20462,
            ("lbs", "kg"):            lambda x: x * 0.453592,
            ("celsius", "fahrenheit"): lambda x: (x * 9/5) + 32,
            ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
        }

        convert = conversions.get((from_unit, to_unit))
        if convert:
            result = convert(value)
            return f"{value} {from_unit} = {result:.4f} {to_unit}"
        return f"Unsupported conversion: {from_unit} to {to_unit}"
    except Exception as e:
        return f"Conversion error: {str(e)}"


# ── Set up the Agent ─────────────────────────────────────────────
llm   = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)
tools = [scientific_calculator, unit_converter]

# Pull the standard ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")

# Create the ReAct agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Wrap in executor — handles the Thought → Action → Observation loop
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,       # prints the full ReAct loop
    max_iterations=6    # safety limit
)

# ── Run the Agent ────────────────────────────────────────────────
# Agent will decide to use calculator AND unit_converter automatically
result = agent_executor.invoke({
    "input": "What is sqrt(256)? Also convert 37 celsius to fahrenheit."
})

print("\n=== Final Answer ===")
print(result["output"])

---
## Component 6 — Tools

**What it is:** Built-in or custom functions the Agent can call. LangChain has 50+ built-in tools.  
**Why it exists:** LLMs generate text only — Tools give them access to the real world (web, databases, code execution).

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ── Tool 1: DuckDuckGo Web Search ────────────────────────────────
search_tool = DuckDuckGoSearchRun()
search_result = search_tool.run("What is LangChain used for?")
print("=== DuckDuckGo Search Result ===")
print(search_result[:400], "...\n")  # print first 400 chars

# ── Tool 2: Wikipedia Query ──────────────────────────────────────
wiki_api     = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=400)
wiki_tool    = WikipediaQueryRun(api_wrapper=wiki_api)
wiki_result  = wiki_tool.run("Large Language Model")
print("=== Wikipedia Result ===")
print(wiki_result)

---
## Component 7 — Document Loaders

**What it is:** Loaders ingest data from various sources (PDFs, web pages, CSVs, Notion etc.) into LangChain Document objects.  
**Why it exists:** RAG pipelines need to ingest external knowledge from diverse formats into a unified structure.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Load a Webpage ───────────────────────────────────────────────
# We load the LangChain docs introduction page
web_loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")
docs = web_loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Source: {docs[0].metadata.get('source', 'N/A')}")
print(f"Content preview (first 300 chars):\n{docs[0].page_content[:300]}")

# ── Split into Chunks ────────────────────────────────────────────
# Splitting is needed before embedding — LLMs have token limits
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # max characters per chunk
    chunk_overlap=50,     # overlap between chunks to preserve context
    separators=["\n\n", "\n", ".", " "]  # split order priority
)
chunks = splitter.split_documents(docs)

print(f"\nSplit into {len(chunks)} chunks")
print(f"Example chunk:\n{chunks[0].page_content[:200]}")

---
## Component 8 — Vector Stores and Indexes

**What it is:** Stores document embeddings and enables fast semantic similarity search.  
**Why it exists:** This is the backbone of RAG — retrieve only the most relevant chunks for a query instead of passing everything to the LLM.

### Architecture Flow:
```
Documents → Embeddings Model → FAISS Vector Store → Similarity Search → Relevant Chunks
```

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ── Sample Knowledge Base ────────────────────────────────────────
documents = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="Agents in LangChain can call tools and reason autonomously using ReAct."),
    Document(page_content="Vector stores enable semantic similarity search over documents."),
    Document(page_content="Memory in LangChain preserves conversation history across turns."),
    Document(page_content="LCEL uses the pipe operator to connect components in a chain."),
    Document(page_content="RAG stands for Retrieval Augmented Generation."),
    Document(page_content="FAISS is a Facebook AI library for fast vector similarity search."),
]

# ── Create Embeddings ────────────────────────────────────────────
# Embeddings convert text → dense numerical vectors (e.g. 1536-dim for OpenAI)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# ── Build FAISS Vector Store ─────────────────────────────────────
vectorstore = FAISS.from_documents(documents, embeddings)
print("Vector store created successfully!")

# ── Perform Similarity Search ────────────────────────────────────
query = "How do agents work in LangChain?"
results = vectorstore.similarity_search(query, k=2)  # top 2 most similar

print(f"\n=== Top 2 Results for: '{query}' ===")
for i, doc in enumerate(results, 1):
    print(f"Result {i}: {doc.page_content}")

# ── Save and Reload (for production use) ────────────────────────
vectorstore.save_local("faiss_index")
print("\nVector store saved to 'faiss_index/' folder.")

loaded_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
print("Vector store reloaded successfully!")

---
## Hands-on Example — Complete RAG Pipeline (Document Q&A)

This combines: Document loading + Text splitting + Embeddings + Vector Store + LCEL Chain  
This is the most common real-world LangChain pattern.

### Architecture Flow:
```
Docs → Chunks → Embed → FAISS Store → Retriever → Prompt (with context) → LLM → Answer
```

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

# ── Step 1: Knowledge Base ───────────────────────────────────────
knowledge_base = [
    Document(page_content="LangChain supports over 50 LLM providers including OpenAI, Anthropic, HuggingFace."),
    Document(page_content="LCEL (LangChain Expression Language) uses the | pipe operator to chain components."),
    Document(page_content="Agents use ReAct: they Reason about the task, then Act by calling a tool, then Observe the result."),
    Document(page_content="FAISS is Facebook AI Similarity Search — a library for fast nearest-neighbor search over vectors."),
    Document(page_content="ConversationBufferMemory stores all messages. WindowMemory keeps only the last k turns."),
    Document(page_content="RAG stands for Retrieval Augmented Generation — retrieves relevant docs before generating an answer."),
    Document(page_content="PromptTemplate allows dynamic prompt construction with named input variables."),
]

# ── Step 2: Embed and Index ──────────────────────────────────────
embeddings  = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
vectorstore = FAISS.from_documents(knowledge_base, embeddings)

# Retriever: fetches top-2 most relevant chunks for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ── Step 3: RAG Prompt ───────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""
You are a LangChain expert assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say: "I don't have that information."

Context:
{context}

Question: {question}

Answer:
""")

# ── Step 4: Build RAG Chain ──────────────────────────────────────
llm    = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)
parser = StrOutputParser()

def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL RAG chain
rag_chain = (
    {
        "context":  retriever | format_docs,   # retrieve → join into string
        "question": RunnablePassthrough()       # pass question unchanged
    }
    | rag_prompt
    | llm
    | parser
)

# ── Step 5: Ask Questions ────────────────────────────────────────
questions = [
    "What operator does LCEL use?",
    "How does memory work in LangChain?",
    "What is FAISS and what is it used for?",
    "What does RAG stand for?"
]

print("=== RAG Pipeline Q&A ===")
for q in questions:
    answer = rag_chain.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")

---
## Real-World Use Case 1 — Customer Support Chatbot with Memory

**Problem:** A company needs a support bot that remembers context within a session.  
**Components used:** ChatOpenAI, ConversationBufferWindowMemory, ConversationChain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationChain
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3, api_key=OPENAI_API_KEY)

# Window memory — keeps only last 5 turns to control token cost
memory = ConversationBufferWindowMemory(k=5)

# Custom system-level context injected via PromptTemplate
support_template = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are a friendly customer support agent for TechCorp.
Help users with billing, account, and product questions.
Always be polite. Escalate to human agents for complex legal issues.

Conversation so far:
{history}

Customer: {input}
Support Agent:"""
)

support_bot = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=support_template,
    verbose=False
)

# Simulate a realistic support session
turns = [
    "Hi, my name is Pranav and I can't log in to my account.",
    "I tried resetting my password but the email never arrives.",
    "My email is pranav@example.com — can you check?",
    "How long will this take to fix?"
]

print("=== Customer Support Chatbot ===")
for turn in turns:
    response = support_bot.predict(input=turn)
    print(f"\nCustomer: {turn}")
    print(f"Bot: {response}")

---
## Real-World Use Case 2 — RAG Document Q&A Assistant

**Problem:** Users want to query a knowledge base (e.g. company docs, policies) in natural language.  
**Components used:** WebBaseLoader, RecursiveCharacterTextSplitter, OpenAIEmbeddings, FAISS, LCEL Chain

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def build_rag_assistant(url: str, api_key: str):
    """
    Builds a RAG Q&A assistant over any webpage.
    Steps: Load → Split → Embed → Store → Retrieve → Answer
    """
    # 1. Load the webpage
    print(f"Loading content from: {url}")
    loader = WebBaseLoader(url)
    docs   = loader.load()

    # 2. Split into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    chunks   = splitter.split_documents(docs)
    print(f"Split into {len(chunks)} chunks")

    # 3. Embed and store
    embeddings  = OpenAIEmbeddings(api_key=api_key)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
    print("Vector store built successfully!")

    # 4. RAG prompt
    prompt = ChatPromptTemplate.from_template("""
    You are a helpful assistant. Use ONLY the context below to answer.
    If not found in context, say 'Not found in the document.'

    Context:
    {context}

    Question: {question}
    Answer:
    """)

    # 5. Build chain
    llm   = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=api_key)
    chain = (
        {"context": retriever | (lambda d: "\n\n".join(x.page_content for x in d)),
         "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain

# Build assistant over LangChain docs
rag_assistant = build_rag_assistant(
    url="https://python.langchain.com/docs/introduction/",
    api_key=OPENAI_API_KEY
)

# Ask questions
test_questions = [
    "What is LangChain?",
    "What are the main components of LangChain?"
]

print("\n=== RAG Document Assistant ===")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_assistant.invoke(q)}")

---
## Real-World Use Case 3 — Autonomous Data Analysis Agent

**Problem:** Analyze a dataset using natural language without writing Pandas code manually.  
**Components used:** ChatOpenAI, create_pandas_dataframe_agent, Python REPL Tool

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.agents import create_pandas_dataframe_agent
import pandas as pd

# ── Sample Sales Dataset ─────────────────────────────────────────
df = pd.DataFrame({
    "month":    ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "sales":    [12000, 15000, 11000, 18000, 22000, 19000],
    "region":   ["North", "North", "South", "South", "East", "East"],
    "expenses": [8000,  9000,  7500,  11000, 13000, 12000]
})
df["profit"] = df["sales"] - df["expenses"]

print("=== Dataset ===")
print(df.to_string(index=False))

# ── Create Pandas Agent ──────────────────────────────────────────
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)

pandas_agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    allow_dangerous_code=True  # required to allow code execution
)

# ── Natural Language Queries over the DataFrame ──────────────────
queries = [
    "What is the total sales across all months?",
    "Which month had the highest profit?",
    "What is the average sales per region?"
]

print("\n=== Data Analysis Agent ===")
for query in queries:
    result = pandas_agent.invoke({"input": query})
    print(f"\nQ: {query}")
    print(f"A: {result['output']}")

---
## Summary — LangChain Component Reference

| Component | Class | Purpose |
|---|---|---|
| LLM Wrapper | `ChatOpenAI` | Unified interface to any LLM |
| Prompt | `ChatPromptTemplate` | Dynamic prompt construction |
| Chain | LCEL `|` operator | Multi-step pipelines |
| Memory | `ConversationBufferMemory` | Preserve chat history |
| Agent | `create_react_agent` | Autonomous tool selection |
| Tool | `@tool` decorator | Extend LLM with real-world actions |
| Doc Loader | `WebBaseLoader`, `PyPDFLoader` | Ingest external knowledge |
| Vector Store | `FAISS` | Semantic similarity search |

---

## Architecture Pipeline
```
User Input → Prompt Template → LLM → Chain/Agent → Tool (if needed) → Output Parser → Final Answer
                  ↑                       ↑
               Memory               Vector Store
           (chat history)         (retrieved docs)
```

---
